# 08. The naive tilt: the mandate's tilt held with no optimiser

**Terms used in this notebook.**

| Term | Meaning |
|---|---|
| Market capitalisation | the number of a firm's shares times their price, the firm's size in money |
| Cap-weighted | weighted by market capitalisation |
| Benchmark | the index the portfolio tracks: French's 49 industries, each held in proportion to its market capitalisation at the start of the month |
| Active weight | the portfolio's weight in an industry minus the benchmark's weight in it |
| Active return | the portfolio's return minus the benchmark's return in the same month |
| Covariance matrix | the table of all variances and covariances of the 49 industries' returns |
| Tracking error | the standard deviation of the monthly active return times the square root of 12, a figure per year; realised when measured on the active returns afterwards, forecast when computed from a covariance matrix before the month |
| Basis point | one hundredth of a percentage point; 100 basis points a year of tracking error is 1% a year |
| Tilt | Coal, Oil and Util (utilities) held at no more than half their benchmark weight |
| Turnover cap | an upper limit on the one-way turnover of a rebalance, 2% a month here |
| Mandate | the rules the optimised portfolios obey in this notebook: long-only, each industry outside the tilt within 2 percentage points of its benchmark weight, the turnover cap and the tilt; risk-models-for-index-tracking calls this set "C3 tilt only" |
| Naive tilt | the portfolio this notebook builds: Coal, Oil and Util at exactly half their benchmark weight, and the weight this removes added to the other 46 industries in proportion to their benchmark weights; it uses no estimate of risk |
| Covariance estimator | a method for estimating the covariance matrix from past returns |
| Shrinkage | pulling an estimate part of the way towards a simpler target, which trades a little bias for less estimation error |
| Factor | a return series that moves many assets at once; the six here are the market (Mkt-RF), size (SMB), value (HML), profitability (RMW), investment (CMA) and momentum (Mom) |
| Principal component | a combination of the industries chosen to explain as much of their total variance as one combination can |
| Optimiser | the routine of risk-models-for-index-tracking that holds, each month, the long-only portfolio with the smallest forecast tracking error under the mandate |
| Path | the month-by-month sequence of portfolios of one method, each formed on the data before its month |
| Optimised path | the path the optimiser holds with one covariance estimator |
| One-way turnover | half the sum over industries of the absolute changes in weight at a rebalance; the fraction of the portfolio sold, which is also the fraction bought |
| Drifted weights | the previous month's weights after that month's returns have moved them |
| Index turnover | the one-way turnover the benchmark itself needs from its drifted weights to its new weights |
| Cost level | the cost charged per unit of one-way turnover, 50 or 100 basis points here, applied after the fact |
| Net active return | the active return per year less 12 times the mean one-way turnover times the cost level |
| Information ratio | the active return per year divided by the realised tracking error |
| Standard error | the standard deviation an estimate would show across repeated samples of the same length |
| Block bootstrap | resampling the months in blocks of 12 consecutive months, circular, so that a block may run past the last month into the first; each resample keeps the months of a block together |
| Draw | one resample of the months, 2,000 here |
| Seed | the number that starts the random number generator, fixed in `constants.py` so that every run draws the same resamples |
| Paired test | a test of the difference between two figures measured on the same months; here the test of Ledoit and Wolf (2011) on the variances of two active returns, with a block bootstrap, copied unchanged in `src/bp/source/paired.py` |
| p-value | the probability, if the two tracking errors were equal, of a difference at least as far from zero as the one measured |
| Holm's correction | the adjustment of Holm (1979) to the p-values of a family of comparisons tested together; a difference is statistically significant at the 5 per cent level when its adjusted p-value is below 0.05 |
| Detectable difference | the difference the paired test would find with probability 0.8 at the 5 per cent level, given its standard error |
| Excess return | a return minus the risk-free rate, the return on a one-month US Treasury bill |
| Beta | how much a return moves with another on average; the realised beta is the slope of the portfolio's monthly excess return on the benchmark's |
| Active exposure | the portfolio's beta to a factor minus the benchmark's beta to it, with each industry's betas from a regression on the six factors over the three years of daily returns before the month |
| Attribution | the split of each month's active return into the active exposures times the factors' returns, and a residual |
| Spread | the cap-weighted return of the 46 industries outside the tilt minus the cap-weighted return of Coal, Oil and Util |
| t-ratio | a mean divided by its standard error |
| Band | the largest difference, fixed before the run, at which a rebuilt figure counts as reproducing the recorded one |
| Vintage | the version of Ken French's files on the download date, named by the release of the CRSP database they were built from |
| Provenance | the record of what was downloaded, when, with its checksum and vintage |

**What this notebook does.** An index fund under the mandate holds Coal, Oil and Util at no more than half their benchmark weight and has to put the weight it removes somewhere. Risk-models-for-index-tracking puts it where an optimiser says, using a covariance estimator. This notebook puts it in the simplest place that obeys the tilt, the other 46 industries in proportion to their benchmark weights, and measures what the optimiser's work saves: the naive tilt's realised tracking error beside each of the seven optimised paths, month by month from 1979-07 to 2026-08, with the paired test and Holm's correction across the seven, and the naive tilt's turnover, costs, information ratio, realised beta and attribution beside theirs.

**Where the optimised paths come from.** The seven paths, their monthly active returns and their measures are read from tables of risk-models-for-index-tracking at commit ddbb75c, copied unchanged into `results/202608/source/`, and the code that built them is copied unchanged into `src/bp/source/`, each file with a first line that names its source file and the commit. `tests/test_source_copies.py` checks every copy. Before any figure of the naive tilt is computed, section 1 rebuilds one optimised path from the copied code and requires it to reproduce the recorded tracking error within a band of 0.1 basis point a year.

The seven optimised paths, by their labels in the copied tables. Each holds the mandate, and is labelled "| C3 tilt only" in the tables below.

| Label | What the optimiser used |
|---|---|
| sample, daily 3y | the sample covariance of the last three years of daily returns, scaled to a month |
| ledoit_wolf, daily 3y | the same, shrunk towards a matrix with the same variance for every industry and no correlation, by the method of Ledoit and Wolf (2004b) |
| factor, daily 3y | the covariance implied by a regression of each industry on the six factors over the same three years |
| pca, daily 3y | the covariance implied by the first five principal components of the same three years |
| sample, monthly 120 | the sample covariance of the last 120 monthly returns |
| robust, all five | no single estimator: the weights whose largest forecast tracking error across the five estimates above is smallest |
| RiskMetrics, daily 5y | J.P. Morgan's 1996 RiskMetrics covariance on five years of daily returns, each day's weight halving every year of trading days |

**How the naive tilt is run.** Each month the naive tilt takes that month's benchmark weights, halves Coal, Oil and Util, and scales the other 46 up by one common factor so that the weights sum to one. It starts as the benchmark in 1979-07, as the optimised paths do, rebalances to these weights every month, and its turnover is measured against its own drifted weights. It is not held to the turnover cap, and the months in which it trades more than the cap allows are counted. The construction is `naive_tilt` in `src/bp/naive.py`.

Running time: about half a minute, most of it rebuilding the optimised path in section 1.

In [ ]:
%%writefile src/bp/constants.py
"""
Pre-committed numbers for the study.

Every band, gate, sample boundary and design parameter is defined here, one
per line, with its source beside it. A notebook may read these names; it may
not introduce numbers of its own. The replication bands (section 3) were
fixed on 10 September 2026, before any of the rules' code was
written. The expectations each notebook tests are
stated in that notebook's introduction and were written before its code; where
a level was changed after a run, the comment beside it says so, with the date
and the reason, so that a reader can judge whether the change was a
specification settled by the paper's own text or a number tuned to the results.

DGU throughout means DeMiguel, Garlappi and Uppal (2009), "Optimal Versus
Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?",
Review of Financial Studies 22(5), 1915-1953. Page numbers are the journal's.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":   "F-F_Research_Data_Factors_CSV.zip",        # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "momentum":   "F-F_Momentum_Factor_CSV.zip",              # Mom, monthly from 1927-01
    "factors5":   "F-F_Research_Data_5_Factors_2x3_CSV.zip",  # Mkt-RF, SMB, HML, RMW, CMA, RF from 1963-07
    "ind10":      "10_Industry_Portfolios_CSV.zip",           # DGU "Industry" dataset
    "ind49":      "49_Industry_Portfolios_CSV.zip",           # the mandate's universe; carries firm counts and average size
    "size_bm25":  "25_Portfolios_5x5_CSV.zip",                # DGU "FF-1-factor" and "FF-4-factor" datasets
    # Daily files, for the betas and the copied covariance estimators of notebooks 08 and 09 (sections 7 and 8).
    "ind49_daily":    "49_Industry_Portfolios_daily_CSV.zip",        # value- and equal-weighted daily returns from 1926-07-01
    "factors3_daily": "F-F_Research_Data_Factors_daily_CSV.zip",     # Mkt-RF, SMB, HML, RF daily from 1926-07-01
    "factors5_daily": "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", # Mkt-RF, SMB, HML, RMW, CMA, RF daily from 1963-07-01
    "momentum_daily": "F-F_Momentum_Factor_daily_CSV.zip",           # Mom daily from 1926-11-03
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# ---------------------------------------------------------------------------
# 2. The DGU replication
# ---------------------------------------------------------------------------

DGU_SAMPLE_START = "1963-07"   # DGU Table 2, p. 1918, for every French-sourced dataset
DGU_SAMPLE_END = "2004-11"     # DGU Table 2, p. 1918
DGU_ESTIMATION_WINDOW = 120    # months; DGU section 2, p. 1927. M = 60 is in DGU's appendix and is out of scope.
DGU_RISK_AVERSION = 1.0        # gamma in the CEQ, DGU p. 1929: "the results we report are for the case of gamma = 1"
DGU_TCOST = 0.0050             # proportional cost per unit of one-way turnover; DGU p. 1929: 50 basis points, after Balduzzi and Lynch (1999)
TCOST_SENSITIVITY = 0.0100     # the one cost sensitivity reported: double the base cost
DGU_GMINC_LOWER_BOUND_FRACTION = 0.5   # g-min-c imposes w >= a * 1 with a = 1/(2N); DGU p. 1926
DGU_BS_COV_DOF = "M - N - 2"   # the Bayes-Stein covariance divides by M - N - 2; DGU equation (5), p. 1923

# Settled 22 September 2026, a specification the paper leaves ambiguous. DGU's
# Lagrangian (8), p. 1925, shows the short-sale constraint alone; footnote 22,
# p. 1934, says the constrained rules produce "corner solutions with all wealth
# invested in a single asset", which only happens when the budget constraint
# 1'w = 1 sits inside the optimisation with gamma = 1. With the budget inside,
# mv-c on MKT/SMB/HML gives a Sharpe ratio of 0.1093 against the published
# 0.1084 and a corner solution in 257 of 377 months; without it, 0.2382 and no
# corner. The reading that reproduces the paper's tables is used; the literal
# reading is kept as rules.mv_c_cone for comparison (notebook 03).
DGU_CONSTRAINED_BUDGET_INSIDE = True

# The constrained rules' solver, set on 17 November 2026: each
# constrained rule is a quadratic programme whose budget 1'w = 1 is an equality
# of the problem and whose lower bound is a bound on each weight, solved
# exactly by the active-set method of
# rules._budget_qp, so that no constraint is approximated by a weight and no
# result depends on a setting of a library or of the module. The replication
# tables of notebook 03 are the tables written under this decision and committed
# with it; the extended-sample tables of notebook 07 stay as committed
# (results/202608/README.md records why). The suite checks the optimality
# conditions at the scale of the study's data (tests/test_rules_backtest.py).
DGU_CONSTRAINED_SOLVER_DATE = "2026-11-17"

# The rules replicated: those whose only inputs are the window's sample moments
# (and, for Bayes-Stein, Jorion's prior built from them), plus the value-weighted
# market, whose Sharpe ratio is the same 0.1138 in every French-sourced column
# of Table 3 and so checks that the market series and the sample match the paper's.
DGU_RULES = ("ew", "mv", "bs", "min", "vw", "mv-c", "bs-c", "min-c", "g-min-c")
DGU_IN_SAMPLE_ROW = "mv (in sample)"   # DGU Table 3 row 2: mean-variance with M = T, the upper bound estimation error costs

# Excluded, each with the criterion. They are not built.
DGU_RULES_EXCLUDED = {
    "Bayesian diffuse-prior": "not reported by DGU (Table 1 note, p. 1917)",
    "dm": "Bayesian Data-and-Model (Pastor 2000); needs an asset-pricing prior, outside the moment-based set",
    "mp": "MacKinlay and Pastor (2000) missing-factor model; needs a maximum-likelihood factor structure, outside the moment-based set",
    "mv-min": "Kan and Zhou (2007) three-fund mixture of two rules already built; not part of the pre-specified set",
    "ew-min": "mixture of 1/N and minimum variance; not part of the pre-specified set",
    "multi-prior": "not reported by DGU (Table 1 note, p. 1917)",
}

# The datasets replicated: the four DGU built from French's site (DGU Table 2, p. 1918; Appendix A, pp. 1948-1949).
# Each is monthly excess returns over the T-bill rate from French's factor file.
DGU_DATASETS = {
    "Industry":     {"n_risky": 10, "factors": ("Mkt-RF",), "N": 11, "source": "ind10 + factors3"},
    "MKT/SMB/HML":  {"n_risky": 0,  "factors": ("Mkt-RF", "SMB", "HML"), "N": 3, "source": "factors3"},
    "FF-1-factor":  {"n_risky": 20, "factors": ("Mkt-RF",), "N": 21, "source": "size_bm25 + factors3"},
    "FF-4-factor":  {"n_risky": 20, "factors": ("Mkt-RF", "SMB", "HML", "Mom"), "N": 24, "source": "size_bm25 + factors3 + momentum"},
}
DGU_DATASETS_EXCLUDED = {
    "S&P Sectors": "Roberto Wessels' GICS sector data, not on French's site (DGU Appendix A.1)",
    "International": "MSCI country indices, not on French's site (DGU Appendix A.3)",
    "FF-3-factor": "DGU do not report it: 'very similar to FF-1-factor' (Table 3 note, p. 1931)",
}
DGU_EXCLUDE_LARGEST_SIZE_QUINTILE = True   # DGU footnote 24, p. 1949: the five largest-size portfolios of the 25 are dropped, leaving 20

# ---------------------------------------------------------------------------
# 3. Bands for the replication, written before the code ran
# ---------------------------------------------------------------------------

BAND_SHARPE_ABS = 0.03          # out-of-sample Sharpe ratio within 0.03 of DGU Table 3, per rule and dataset
CHECK_TOP3_RANKING = True        # the ranking of the top three rules within each dataset is preserved
BAND_TURNOVER_REL = 0.25        # turnover within 25% relative of DGU Table 5

# DGU Table 5 reports the unconstrained mean-variance rule's turnover at 606,594
# times that of 1/N on the Industry dataset and 10,466 times on FF-1-factor.
# Those figures are dominated by the months in which the sample covariance is
# near-singular and move by orders of magnitude when French revises a single
# return, so the band applies only where DGU's published relative turnover is
# below the level here; above it, the replicated figure is reported beside the
# published one with no pass or fail. Fixed 10 September 2026.
BAND_TURNOVER_APPLIES_BELOW_RELATIVE = 100.0

# Changed 13 September 2026, after notebook 02 had run, on its evidence: the
# out-of-sample Sharpe band applies only to rules whose published relative
# turnover (DGU Table 5, panel A) is below the same level, and the in-sample
# mean-variance row carries no verdict. Evidence: on the Industry, FF-1 and FF-4
# datasets the unconstrained mean-variance rule moved by 0.12 to 0.25 in Sharpe
# ratio under ten basis points of noise, with the published figure inside the
# range, while the 1/N rows matched to the third decimal. Before the change the
# band applied to every out-of-sample row. The level is the paper's own number,
# already used by the turnover band.
BAND_SHARPE_APPLIES_BELOW_RELATIVE = BAND_TURNOVER_APPLIES_BELOW_RELATIVE
BAND_SHARPE_DECISION_DATE = "2026-09-13"

# Sharpe ratios are compared on the same sample, 1963-07 to 2004-11, 120-month
# window, so the out-of-sample period is 1973-07 to 2004-11.
DGU_EXPECTED_T = 497           # months from 1963-07 to 2004-11 inclusive; a count, checked in notebook 01
DGU_EXPECTED_OOS = 377         # DGU_EXPECTED_T - DGU_ESTIMATION_WINDOW

# The replication extended to the latest month (notebook 07), fixed 1 October
# 2026 before the notebook's code: the nine rules on every month the vintage
# carries, reported for the paper's out-of-sample period, the months since the
# paper and the whole period. The expectations are in the notebook's introduction.
EXTENDED_NEW_PERIOD_START = "2004-12"   # the month after DGU's sample ends

# ---------------------------------------------------------------------------
# 4. DGU's analytical result and simulation (notebook 04)
# ---------------------------------------------------------------------------

# The analytical critical window (DGU Proposition 1, pp. 1937-1938; Figure 1, p. 1940).
# Panel E is the one calibrated to US stock-market data and the one the abstract quotes.
ANALYTIC_SHARPE_TANGENCY = 0.15    # S*, DGU Figure 1 panel E
ANALYTIC_SHARPE_1N = 0.12          # S_ew, DGU Figure 1 panel E
EXPECT_CRITICAL_M_LOWER = {25: 3000, 50: 6000}   # DGU p. 1941: "more than 3000 months" and "more than 6000 months"; the check is order of magnitude and direction in N

# The simulated dataset (DGU section 5.1, pp. 1941-1942).
SIM_RF_MEAN_ANNUAL = 0.02
SIM_RF_SD_ANNUAL = 0.02
SIM_FACTOR_MEAN_ANNUAL = 0.08
SIM_FACTOR_SD_ANNUAL = 0.16
SIM_ALPHA = 0.0
SIM_BETA_RANGE = (0.5, 1.5)          # loadings evenly spread
SIM_IDIO_VOL_RANGE = (0.10, 0.30)    # annual idiosyncratic volatility, uniform, cross-sectional mean 0.20
SIM_N = (10, 25, 50)
SIM_M = (120, 360, 6000)
SIM_T = 24000                        # months in one simulated history
SIM_SEED = 20260910                  # fixed so the run is reproducible; DGU's seed is unknown
# In one 24,000-month history the standard error of a monthly Sharpe ratio is
# about 1/sqrt(24000) = 0.0065, so the comparison with DGU Table 6 uses the same
# band as the empirical tables, plus the sign of (mv minus 1/N) in every cell.
BAND_SIM_SHARPE_ABS = BAND_SHARPE_ABS
# Table 6 reports 1/N and "mv (true)" as full-sample Sharpe ratios and the
# estimated rules out of sample; the comparison follows that convention. As a
# second check every rule is rerun on alternative draws of the market (seeds 1
# to SIM_SPREAD_DRAWS) and the published value is checked against their range.
SIM_SPREAD_DRAWS = 8

# Fat tails (notebook 05), an addition to the paper, fixed 30 September 2026
# before the notebook's code. The market above is rerun with every shock drawn
# from a Student-t distribution, scaled so that the mean and covariance of
# returns are exactly those of the normal market; one scale draw per month is
# shared by all shocks (a multivariate t), so an extreme month hits all assets
# at once. The normal shocks are the same draws as notebook 04's.
SIM_T_DOF = (10, 5)                  # degrees of freedom. 10: excess kurtosis 1, about a broad index's monthly returns. 5: excess kurtosis 6, about an industry portfolio's. At 4 or below the fourth moment is infinite and the standard error of a Sharpe ratio is undefined.
SIM_T_COMMON_SCALE = True            # one scale per month shared by all shocks (multivariate t); False draws one scale per shock series, which changes the shape of the month's co-movement as well as its size
# The crossing table: sample-based mean-variance against 1/N over a grid of
# windows, to locate the window at which the gap closes under each distribution.
SIM_CROSSING_WINDOWS = (120, 240, 360, 600, 1000, 2000, 3000, 4000, 6000, 9000, 12000)
FAT_TAIL_MOMENT_CHECK_ABS = 0.013    # notebook 05, expectation 1: two standard errors of a 24,000-month Sharpe ratio
FAT_TAIL_DIRECTION_SLACK = 0.01      # notebook 05, expectation 2: a cell counts as in the expected direction if Sharpe(t) <= Sharpe(normal) + this
# One test added after notebook 05's first run (6 October 2026), fixed here before
# its code and labelled in the notebook: the fat-tailed market rerun with one
# scale per shock series (common_scale False), so that an extreme month changes
# the shape of the estimated covariance as well as its level. Expectation P1a:
# the mean change in Sharpe ratio across the seven estimated rules and the nine
# cells of the Table 6 grid is more negative than under the common scale at both
# degrees of freedom. P1b: the rules whose weights depend on the shape of the
# covariance and use no means lose more on average than the other estimated rules.
FAT_TAIL_SHAPE_RULES = ("min", "min-c", "g-min-c")   # P1b: the minimum-variance rules

# Volatility that changes through time (notebook 06), the second addition to
# the paper, fixed 1 October 2026 before the notebook's code. The market of
# notebook 04 is rerun with shocks whose volatility follows a GARCH(1,1) process
# (Bollerslev 1986); the factor shock and every asset's idiosyncratic shock get
# their own process with the same parameters, so both the level and the shape
# of the covariance matrix move through time, while the unconditional mean and
# covariance are exactly those of the normal market (omega = 1 - alpha - beta).
# The expectations of notebooks 05 and 06 are in their introductions and use
# the two levels above.
SIM_GARCH_ALPHA = 0.10               # weight of last month's squared shock; a textbook monthly value
SIM_GARCH_BETA = 0.85                # weight of last month's variance; alpha + beta = 0.95 is the persistence, a half-life of a volatility shock of about 13 months
SIM_GARCH_OWN_PROCESS_PER_ASSET = True   # the factor and each idiosyncratic shock follow their own process; False would scale all shocks by one common volatility, which leaves the shape of the covariance unchanged

# ---------------------------------------------------------------------------
# 5. The 49 industries: the sample, and the three industries the mandate tilts
# ---------------------------------------------------------------------------

# The mandate's sample starts at the first month in which all 49 industries carry
# a return (French codes the early gaps as -99.99): Hlth is the last industry to
# fill and does so in 1969-07. Notebook 01 asserts it on every run.
EXT_SAMPLE_START = "1969-07"
# The end of the sample, fixed on 6 November 2026. Until then the end was the
# latest month the vintage carried, and every recorded run (7 to 18 October
# 2026) used the 202608 vintage, whose last month is 2026-08, the end the design
# names ("1979-07 to 2026-08"). French's library moved to the 202609 vintage
# before 6 November, which carries 2026-09 and so gave 567 evaluation months
# and stopped every notebook at its count. From now on the loader cuts every
# file at this month, whatever the vintage, so that each notebook sees the same
# months on any later vintage; the vintage line and the SHA-256 in the
# provenance record say which revision of French's history a run used, and a
# later vintage need not reproduce every digit of the committed tables, since
# French revises history. A run on the cached 202608 files is unchanged by the
# cut. Not an expectation: a data rule, disclosed here and in the README.
EXT_SAMPLE_END = "2026-08"
# The rule, written here on 8 November 2026 after the decision of 6 November, and carried in the README's run section in
# the same words: the sample is July 1979 to August 2026, fixed in the code.
# The recorded tables are those of the 202608 vintage as cached. A run on a
# later vintage, cut at August 2026, is a confirmation run: it is labelled by
# its vintage in the provenance record, it is expected to reproduce every
# verdict line and every printed decimal and not every digit, because French
# revises history, and it replaces no committed table without a dated decision.
EXT_SAMPLE_END_RULE_DATE = "2026-11-08"
EXT_CONFIRMATION_RUN = "a run on a later vintage, cut at August 2026, labelled by its vintage in the provenance record"

TILT_INDUSTRIES = ("Coal", "Oil", "Util")   # French's column names, stripped of padding

# ---------------------------------------------------------------------------
# 6. Output conventions
# ---------------------------------------------------------------------------

OUTPUT_DIR = "outputs"
PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

# ---------------------------------------------------------------------------
# 7. The code and the tables taken from risk-models-for-index-tracking
# ---------------------------------------------------------------------------

# The comparisons with the naive portfolios under the index mandate run the optimised portfolios' own code and read their
# recorded figures. Both come from the repository risk-models-for-index-tracking at commit ddbb75c, whose history up to
# that commit is this repository's. The modules are copied into SOURCE_MODULES_DIR, each unchanged below a first line
# naming its source file and the commit, and src/bp/source/constants.py holds the source's values of the names they
# read. The tables are copied unchanged into SOURCE_TABLES_DIR. tests/test_source_copies.py checks every copy against
# the SHA-256 below and, in a checkout that holds the commit, against the file at the commit.
SOURCE_REPOSITORY = "https://github.com/ochofer/risk-models-for-index-tracking"
SOURCE_COMMIT = "ddbb75c"
SOURCE_MODULES_DIR = "src/bp/source/"
SOURCE_TABLES_DIR = "results/202608/source/"
SOURCE_MODULES = {
    "benchmark.py": ("src/bp/benchmark.py", "f9c6b71a8f44a66e2756e18d75281729fc4cd65e598a420384661e149d9df2b1"),
    "covariance.py": ("src/bp/covariance.py", "296b842291ad829c48fa784db114adc3495f26cb1441cc3776357421280b193a"),
    "optimiser.py": ("src/bp/optimiser.py", "8944b967bb5969fe3c4b2ffbc603aa2b75a6fddc0683735fc842ec241e303b93"),
    "evaluation.py": ("src/bp/evaluation.py", "5e92444e694996f7801864ce3feaea96a2d290aae97ac2e1525e78229d59d23d"),
    "attribution.py": ("src/bp/attribution.py", "61e14a7106f9398f5d82a178f376dc3fa8ec73fa48baeacff91c1dcd53f0c97c"),
    "paired.py": ("src/bp/paired.py", "579d216032e9ec6ab8e3c0619dd5c33700afbe1a873d2cfe9ea454c01a264bb5"),
    "rules.py": ("src/bp/rules.py", "c18872058185839981806dafdc5de417b25ea4a9d2e607cd01d615775a4addeb"),
    "french_loader.py": ("src/bp/french_loader.py", "83e01cb19c5640d81e19fe828b816f5bd115ea15e3b56fdbd63dea8aaabca6b1"),
}
SOURCE_CONSTANTS_SHA256 = "0ba29dc41644607d89318d50c16299901453b4e89b4444f119982fbd8912b4d9"   # src/bp/constants.py at the commit, 1,168 lines
SOURCE_TABLES = {
    "rolling_evaluation_202608.csv": ("results/202608/rolling_evaluation_202608.csv", "32cea22bcf34f975923f60af457bf653c731d4cdc71043e6166b944cfd279446"),
    "active_returns_202608.csv": ("results/202608/active_returns_202608.csv", "f83481400080e5cbaa8a4fb907e72a59ec97de41b89c6c89f8185d2cc51cd2c4"),
    "attribution_summary_202608.csv": ("results/202608/attribution_summary_202608.csv", "6059b492760f4cee547555b0a52558134710d0a356de5197d98cbd7968d1c877"),
    "factor_objective_evaluation_202608.csv": ("results/202608/factor_objective_evaluation_202608.csv", "d9bc6c735a350942a49fecce64595190a7c2280e2bc5057835eba0c729c1a926"),
    "factor_objective_monthly_202608.csv": ("results/202608/factor_objective_monthly_202608.csv", "a19f45ea671af1553912446a1bc848bae918e07e75d3509f706c117e1a99ef2b"),
    "factor_objective_attribution_202608.csv": ("results/202608/factor_objective_attribution_202608.csv", "7b917c8ebabd805f47526a97d351e7ade83351ac2d2cf73b3d69f426d9bacd4c"),
}

# ---------------------------------------------------------------------------
# 8. The comparisons with the naive portfolios under the index mandate (notebooks 08 and 09)
# ---------------------------------------------------------------------------

# Each name below that also stands in src/bp/constants.py at SOURCE_COMMIT carries the value it has there, so that the
# naive portfolios meet the optimised paths on the same mandate, the same months and the same measures;
# tests/test_source_copies.py checks them against the file at the commit (NAIVE_TAKEN_FROM_SOURCE).
COV_EVAL_START = "1979-07"            # the first month of every path: 1969-07 plus the 120 months of the monthly sample covariance
EVAL_START_PORTFOLIO = "benchmark"    # every path, optimised or naive, starts as the benchmark
ACTIVE_WEIGHT_BOUND = 0.02            # each untilted industry within 2 percentage points of its benchmark weight
TURNOVER_CAP_MONTHLY_ONE_WAY = 0.02   # the optimised paths' cap on one-way turnover a month, against drifted weights; the naive portfolios are measured against it and are not held to it
TILT_MAX_SHARE_OF_BENCHMARK = 0.5     # Coal, Oil and Util at no more than half their benchmark weight; the naive portfolios hold them at exactly half
MANDATE_EXCLUSIONS = ("Smoke", "Guns")   # tobacco and weapons, weight zero under the full mandate and under the factor objective
CONSTRAINT_SET_TILT_ONLY = ("long_only", "active_weight_bound", "turnover_cap", "tilt")   # the optimised paths the naive tilt meets
CONSTRAINT_SETS = {"C3": ("long_only", "active_weight_bound", "turnover_cap", "tilt", "beta_neutral", "exclusion")}   # the mandate, under which the factor objective runs
FO_CONSTRAINT_SET = "C3"
EXT_COST_LEVELS = (DGU_TCOST, TCOST_SENSITIVITY)   # each net return at 50 and at 100 basis points per unit of one-way turnover
EVAL_BOOTSTRAP_BLOCK_MONTHS = 12      # the circular block bootstrap's block, one year
EVAL_BOOTSTRAP_DRAWS = 2000
EVAL_BOOTSTRAP_SEED = 20261009
EVAL_BOOTSTRAP_LEVEL = 0.95
PAIRED_PERIODS = 12                   # months a year, the annualisation of the paired test
PAIRED_DETECTABLE_POWER = 0.80        # the power at which the paired test's detectable difference is stated
ATTRIBUTION_FACTORS = ("Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom")
FO_TARGET_FACTORS = ("HML", "RMW", "CMA", "Mom")   # value, profitability, investment, momentum
FO_TARGET_SIGN = {"Mkt-RF": 0.0, "SMB": 0.0, "HML": 1.0, "RMW": 1.0, "CMA": 1.0, "Mom": 1.0}   # s: the score and the objective sum the four targeted betas
FO_BETA_WINDOW_YEARS = 3              # the betas: a regression of each industry on the six factors over three years of daily returns before the month
FO_BUDGETS_ANNUAL = (0.0075, 0.0100, 0.0150)   # the optimised factor paths' budgets on forecast tracking error, a year
FO_BUDGET_MAIN = 0.0100               # the budget of the seven optimised factor paths the tilted index meets
ATTRIBUTION_REPRODUCTION_TOL = 0.00001          # a rebuilt optimised path reproduces the committed realised tracking error within 0.1 basis point a year
ATTRIBUTION_IDENTITY_MAX_ABS_ERROR = 1e-9       # the factor contributions plus the residual less the active return, per month
NAIVE_TAKEN_FROM_SOURCE = ("COV_EVAL_START", "EVAL_START_PORTFOLIO", "ACTIVE_WEIGHT_BOUND", "TURNOVER_CAP_MONTHLY_ONE_WAY",
                           "TILT_MAX_SHARE_OF_BENCHMARK", "MANDATE_EXCLUSIONS", "CONSTRAINT_SET_TILT_ONLY", "FO_CONSTRAINT_SET",
                           "EXT_COST_LEVELS", "EVAL_BOOTSTRAP_BLOCK_MONTHS", "EVAL_BOOTSTRAP_DRAWS", "EVAL_BOOTSTRAP_SEED",
                           "EVAL_BOOTSTRAP_LEVEL", "PAIRED_PERIODS", "PAIRED_DETECTABLE_POWER", "ATTRIBUTION_FACTORS",
                           "FO_TARGET_FACTORS", "FO_TARGET_SIGN", "FO_BETA_WINDOW_YEARS", "FO_BUDGETS_ANNUAL", "FO_BUDGET_MAIN",
                           "ATTRIBUTION_REPRODUCTION_TOL", "ATTRIBUTION_IDENTITY_MAX_ABS_ERROR")

# The seven optimised paths each naive portfolio meets, by their labels in the copied tables: the sample covariance,
# Ledoit-Wolf shrinkage, the six-factor model and five principal components on three years of daily returns; the sample
# covariance on 120 months; the robust portfolio across all five; RiskMetrics on five years of daily returns.
NAIVE_PATHS = ("sample, daily 3y", "ledoit_wolf, daily 3y", "factor, daily 3y", "pca, daily 3y",
               "sample, monthly 120", "robust, all five", "RiskMetrics, daily 5y")
NAIVE_REPRODUCED = "sample, daily 3y"   # the path each notebook rebuilds from the copied modules before any naive figure
NAIVE_SIGNIFICANCE = 0.05             # a difference is statistically significant when its Holm-adjusted p-value is below this

In [ ]:
%%writefile src/bp/naive.py
"""The two naive portfolios under the index mandate, and the measures that set them beside the optimised paths (notebooks 08 and 09).

A naive portfolio takes its weights from the benchmark weights alone, or from the benchmark weights and one score per
industry, with no covariance estimate:

- naive_tilt: Coal, Oil and Util held at half their benchmark weight, and the weight this removes spread over the
  other 46 industries in proportion to their benchmark weights (notebook 08).
- tilted_index: the same tilt with tobacco and weapons excluded, the removed weight spread over the other 44, and each
  of the 44 then scaled by one plus a common multiple k of its centred factor score, k as large as the bound on each
  active weight and the floor at zero allow (notebook 09).

run_path holds a naive portfolio month by month as evaluation.run_path holds an optimised one: the path starts as the
benchmark, rebalances to the month's weights every month, and measures its one-way turnover against its previous weights
drifted by the previous month's returns. The remaining functions compute the measures the two notebooks report.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from .source import attribution as A
from .source import evaluation as E
from .source import optimiser as O


def naive_tilt(b: np.ndarray, tilt: np.ndarray) -> tuple[np.ndarray, dict]:
    """The naive tilt's weights for one month, from the benchmark weights b; tilt marks Coal, Oil and Util."""
    w = b.copy()
    removed = (1.0 - C.TILT_MAX_SHARE_OF_BENCHMARK) * b[tilt].sum()
    w[tilt] = C.TILT_MAX_SHARE_OF_BENCHMARK * b[tilt]
    rest = ~tilt
    w[rest] = b[rest] * (1.0 + removed / b[rest].sum())
    return w, {"removed weight": removed}


def tilted_index(b: np.ndarray, tilt: np.ndarray, excluded: np.ndarray, score: np.ndarray,
                 names: list[str]) -> tuple[np.ndarray, dict]:
    """The tilted index's weights for one month.

    b: the benchmark weights; tilt and excluded: the masks of the tilted and the excluded industries; score: each
    industry's score, the sum of its betas to the four targeted factors; names: the industries' names, in b's order.
    Returns the weights and k, the constraint that set it ("bound" or "floor") and the industry at which it binds.
    """
    rest = ~(tilt | excluded)
    w0 = b.copy()
    w0[tilt] = C.TILT_MAX_SHARE_OF_BENCHMARK * b[tilt]
    w0[excluded] = 0.0
    removed = (1.0 - C.TILT_MAX_SHARE_OF_BENCHMARK) * b[tilt].sum() + b[excluded].sum()
    w0[rest] = b[rest] * (1.0 + removed / b[rest].sum())
    s, w_rest, b_rest = score[rest], w0[rest], b[rest]
    z = s - (w_rest @ s) / w_rest.sum()           # the score centred on its w0-weighted mean over the 44
    a = w_rest - b_rest                            # the active weight of w0, zero or more for each of the 44
    c = w_rest * z                                 # the change in active weight per unit of k
    rest_names = [n for n, keep in zip(names, rest) if keep]
    limits = []
    for i in range(len(z)):
        if c[i] > 0:
            limits.append(((C.ACTIVE_WEIGHT_BOUND - a[i]) / c[i], "bound", rest_names[i]))
        elif c[i] < 0:
            limits.append(((a[i] + C.ACTIVE_WEIGHT_BOUND) / -c[i], "bound", rest_names[i]))
            limits.append((1.0 / -z[i], "floor", rest_names[i]))
    k, source, name = min(limits, key=lambda x: x[0])
    w = w0.copy()
    w[rest] = w_rest * (1.0 + k * z)
    return w, {"removed weight": removed, "k": k, "k set by": source, "k set at": name,
               "largest active weight of w0": float(np.abs(w0 - b)[rest].max())}


def run_path(weights_of, months: pd.PeriodIndex, weights: pd.DataFrame, returns_m: pd.DataFrame, betas: dict,
             factors: list[str], sign: np.ndarray) -> pd.DataFrame:
    """One naive path: weights_of(month, b) gives the month's weights and a dict of descriptions.

    Each month records the active return, the one-way turnover of the path and of the benchmark, the largest active
    weight, the effective number and the industries held, the six active exposures with that month's betas, and the
    objective, the exposures summed with the signs in sign. The weights go into the column "weights".
    """
    recs = []
    w_prev, prev = None, None
    for m in months:
        b = weights.loc[m].to_numpy()
        r = returns_m.loc[m].to_numpy()
        w, info = weights_of(m, b)
        if w_prev is None:
            w0, b_drift = b.copy(), b.copy()
        else:
            w0 = O.drifted_weights(w_prev, returns_m.loc[prev].to_numpy())
            b_drift = O.drifted_weights(weights.loc[prev].to_numpy(), returns_m.loc[prev].to_numpy())
        x = A.active_exposures(betas[m], w, b)
        recs.append({"month": m, "active_return": float(w @ r - b @ r), "turnover": O.one_way_turnover(w, w0),
                     "benchmark_turnover": O.one_way_turnover(b, b_drift), "largest active weight": float(np.abs(w - b).max()),
                     "effective_number": O.effective_number(w), "held": int((w > 1e-6).sum()), "objective": float(sign @ x),
                     **{f"exposure {f}": float(v) for f, v in zip(factors, x)}, **info, "weights": w})
        w_prev, prev = w, m
    return pd.DataFrame(recs).set_index("month")


def realised_beta(active, bench_ex) -> float:
    """The slope of the portfolio's monthly excess return (the benchmark's plus the active return) on the benchmark's."""
    bench_ex = np.asarray(bench_ex, dtype=float)
    port_ex = bench_ex + np.asarray(active, dtype=float)
    return float(np.cov(port_ex, bench_ex, ddof=1)[0, 1] / np.var(bench_ex, ddof=1))


def measures(active, turnover, bench_ex, objective=None, exposures=None) -> dict:
    """Realised tracking error, gross active return, turnover, cost and net active return at each cost level, the
    information ratios, the realised beta, and, given the monthly objective and the monthly exposures (a table with a
    column "exposure F" for each factor F), the exposure per basis point and the mean exposure to each factor."""
    active = np.asarray(active, dtype=float)
    te = E.realised_tracking_error(active)
    gross = 12.0 * float(active.mean())
    out = {"realised TE": te, "active return (gross)": gross, "turnover (one-way)": float(np.mean(turnover)),
           "information ratio (gross)": gross / te, "realised beta": realised_beta(active, bench_ex)}
    for c in C.EXT_COST_LEVELS:
        bp = int(round(c * 1e4))
        cost = 12.0 * float(np.mean(turnover)) * c
        out[f"cost at {bp} bp"] = cost
        out[f"net active return at {bp} bp"] = gross - cost
        out[f"information ratio at {bp} bp"] = (gross - cost) / te
    if objective is not None:
        out["mean objective"] = float(np.mean(objective))
        out["exposure per bp of realised TE"] = float(np.mean(objective)) / (1e4 * te)
    if exposures is not None:
        for f in C.ATTRIBUTION_FACTORS:
            out[f"mean exposure {f}"] = float(np.mean(exposures[f"exposure {f}"]))
    return out


def resample_indices(n: int) -> np.ndarray:
    """The circular block resamples of evaluation.block_bootstrap_band: ceil(n / block) block starts per resample, each block
    running past the last month into the first, the generator seeded with EVAL_BOOTSTRAP_SEED."""
    block, draws = C.EVAL_BOOTSTRAP_BLOCK_MONTHS, C.EVAL_BOOTSTRAP_DRAWS
    rng = np.random.default_rng(C.EVAL_BOOTSTRAP_SEED)
    k = int(np.ceil(n / block))
    starts = rng.integers(0, n, size=(draws, k))
    return ((starts[:, :, None] + np.arange(block)[None, None, :]) % n).reshape(draws, k * block)[:, :n]


def exposure_per_bp(objective: np.ndarray, active: np.ndarray) -> np.ndarray:
    """The mean objective over 10,000 times the realised tracking error, along the last axis."""
    return objective.mean(axis=-1) / (1e4 * active.std(axis=-1, ddof=1) * np.sqrt(12.0))


def bootstrap_difference(d_star: np.ndarray) -> tuple[float, float, float]:
    """The 95% percentile interval of the resampled differences and the two-sided p-value: twice the smaller of the shares
    at or below zero and at or above zero, each counted as (count + 1) / (draws + 1), at most 1."""
    tail = 100.0 * (1.0 - C.EVAL_BOOTSTRAP_LEVEL) / 2.0
    lo, hi = np.percentile(d_star, [tail, 100.0 - tail])
    draws = len(d_star)
    p = min(1.0, 2.0 * min((int((d_star <= 0).sum()) + 1) / (draws + 1), (int((d_star >= 0).sum()) + 1) / (draws + 1)))
    return float(lo), float(hi), p


def information_ratios(active, costs: dict, idx: np.ndarray) -> dict:
    """The information ratio gross and net of each cost, each with the standard deviation of the ratio over the resamples
    idx; the cost is held at the full-sample figure, 12 times the mean one-way turnover times the cost level."""
    a = np.asarray(active, dtype=float)
    a_star = a[idx]
    te_star = a_star.std(axis=1, ddof=1) * np.sqrt(12.0)
    g_star = 12.0 * a_star.mean(axis=1)
    te = E.realised_tracking_error(a)
    row = {"information ratio (gross)": 12.0 * a.mean() / te, "standard error (gross)": float((g_star / te_star).std(ddof=1))}
    for c, cost in costs.items():
        row[f"information ratio at {c} bp"] = (12.0 * a.mean() - cost) / te
        row[f"standard error at {c} bp"] = float(((g_star - cost) / te_star).std(ddof=1))
    return row

In [ ]:
import os
import sys
import json
import time
if not (os.path.isdir("src/bp/source") and os.path.isdir("results/202608/source")):
    raise SystemExit("This notebook runs from the root of a checkout of this repository, which holds the modules and tables copied from risk-models-for-index-tracking.")
sys.path.insert(0, "src")
import numpy as np
import pandas as pd
from bp import constants as C
from bp import french_loader as fl
from bp import naive as N
from bp.source import attribution as A
from bp.source import covariance as CV
from bp.source import evaluation as E
from bp.source import optimiser as O
from bp.source import paired as PT

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)

records = [fl.download(key) for key in ("ind49", "factors3", "factors5", "momentum", "ind49_daily", "factors3_daily", "factors5_daily", "momentum_daily")]
fl.write_provenance(records)
vintage = records[0]["vintage_line"].split("using the ")[1].split(" ")[0]
assert len({r["vintage_line"].split("using the ")[1].split(" ")[0] for r in records}) == 1, "the eight files are of one vintage"
committed = {r["filename"]: r["zip_sha256"] for r in json.load(open("results/202608/provenance_french.json"))}
same = all(committed.get(r["filename"]) == r["zip_sha256"] for r in records)
print("vintage:", vintage, "| the eight files", "are the committed ones" if same else "differ from the committed ones", "| provenance written to", C.PROVENANCE_FILE)


def source_table(stem):
    name = [k for k in C.SOURCE_TABLES if k.startswith(stem + "_")][0]
    return pd.read_csv(C.SOURCE_TABLES_DIR + name), name


def out(stem):
    os.makedirs(C.OUTPUT_DIR, exist_ok=True)
    return f"{C.OUTPUT_DIR}/{stem}_{vintage}.csv"


P = CV.Panels()
weights, returns_m = P.weights, P.returns_m
months = pd.period_range(C.COV_EVAL_START, P.last, freq="M")
assert len(months) == 566 and str(months[0]) == "1979-07" and str(months[-1]) == C.EXT_SAMPLE_END
industries = [c.strip() for c in weights.columns]
tilt_mask, _ = O.masks(weights.columns)
_, mandate_mask = O.masks(weights.columns, exclude=C.MANDATE_EXCLUSIONS)
FACTORS = list(C.ATTRIBUTION_FACTORS)
TARGETED = list(C.FO_TARGET_FACTORS)
S = np.array([C.FO_TARGET_SIGN[f] for f in FACTORS])
bench_ex = P.bench_m.loc[months].to_numpy()
COSTS = [int(round(c * 1e4)) for c in C.EXT_COST_LEVELS]

t0 = time.time()
Sigma, betas = {}, {}
for m in months:
    Sigma[m], _, _ = CV.estimate("sample", "daily_3y", m, P)
    X = CV.daily_window(P.excess_d, m, C.FO_BETA_WINDOW_YEARS)
    _, betas[m] = CV.factor_cov(X.to_numpy(), P.factors_d.loc[X.index, FACTORS].to_numpy())
IDX = N.resample_indices(len(months))
print(f"{len(months)} months, {months[0]} to {months[-1]}: the sample covariance and the betas on three years of daily returns in {time.time() - t0:.0f}s")

## 1. One optimised path rebuilt from the copied code

The path of the sample covariance on three years of daily returns is rebuilt month by month with the copied optimiser and compared with the realised tracking error and the monthly active returns recorded in the copied tables. The notebook stops here unless the difference is inside the band.

In [ ]:
rolling, rolling_name = source_table("rolling_evaluation")
tilt_rows = rolling[rolling["set"] == "C3 tilt only"].set_index("variant").loc[list(C.NAIVE_PATHS)]
active_paths, _ = source_table("active_returns")
active_paths.index = pd.PeriodIndex(active_paths["month"], freq="M")
label = f"{C.NAIVE_REPRODUCED} | C3 tilt only"
t0 = time.time()
rebuilt = E.run_path(months, Sigma, weights, returns_m, C.CONSTRAINT_SET_TILT_ONLY, tilt_mask, None)
te_rebuilt = E.realised_tracking_error(rebuilt["active_return"])
te_recorded = float(tilt_rows.loc[C.NAIVE_REPRODUCED, "realised TE"])
reproduction = pd.DataFrame([{
    "path": label, "committed table": rolling_name, "rebuilt realised TE (bp)": 1e4 * te_rebuilt,
    "committed realised TE (bp)": 1e4 * te_recorded, "difference (bp)": 1e4 * (te_rebuilt - te_recorded),
    "largest monthly active return difference": float(np.abs(rebuilt["active_return"].to_numpy() - active_paths.loc[months, label].to_numpy()).max()),
    "band (bp)": 1e4 * C.ATTRIBUTION_REPRODUCTION_TOL}])
reproduction["passes"] = reproduction["difference (bp)"].abs() <= reproduction["band (bp)"]
reproduction.to_csv(out("naive_tilt_reproduction"), index=False, float_format="%.8f")
print(f"rebuilt in {time.time() - t0:.0f}s")
print(reproduction.T.to_string())
assert reproduction["passes"].all(), "the rebuilt path does not reproduce the recorded tracking error, so no figure of the naive tilt is computed"

## 2. The naive tilt, month by month

The cell runs the naive tilt over the 566 months and checks each month's weights: they sum to one, none is negative, Coal, Oil and Util are at half their benchmark weight, and each of the other 46 industries is within the mandate's 2 percentage points of its benchmark weight, a bound the naive tilt is not built to respect and meets in every month.

In [ ]:
tilt = N.run_path(lambda m, b: N.naive_tilt(b, tilt_mask), months, weights, returns_m, betas, FACTORS, S)
W = np.vstack(tilt["weights"].to_numpy())
B = weights.loc[months].to_numpy()
assert np.abs(W.sum(axis=1) - 1.0).max() < 1e-12 and (W >= 0).all()
assert np.abs(W[:, tilt_mask] - C.TILT_MAX_SHARE_OF_BENCHMARK * B[:, tilt_mask]).max() < 1e-15
assert np.abs(W - B)[:, ~tilt_mask].max() <= C.ACTIVE_WEIGHT_BOUND
tilt.drop(columns="weights").to_csv(out("naive_tilt_monthly"), float_format="%.10f")
pd.DataFrame(W, index=tilt.index, columns=industries).to_csv(out("naive_tilt_weights"), float_format="%.10f")
row = N.measures(tilt["active_return"], tilt["turnover"], bench_ex, tilt["objective"], tilt)
row.update({"months above the turnover cap": int((tilt["turnover"] > C.TURNOVER_CAP_MONTHLY_ONE_WAY).sum()),
            "first month turnover": float(tilt["turnover"].iloc[0]), "largest monthly turnover": float(tilt["turnover"].max()),
            "month of the largest turnover": str(tilt["turnover"].idxmax()), "mean benchmark turnover": float(tilt["benchmark_turnover"].mean()),
            "largest active weight over the months": float(tilt["largest active weight"].max()),
            "mean effective number": float(tilt["effective_number"].mean()), "mean industries held": float(tilt["held"].mean())})
measures = pd.DataFrame([row], index=pd.Index(["naive tilt"], name="portfolio"))
measures.to_csv(out("naive_tilt_measures"), float_format="%.10f")
print(f"largest active weight of the 46 industries outside the tilt: {np.abs(W - B)[:, ~tilt_mask].max():.4f}")
print(f"weight removed from Coal, Oil and Util: {tilt['removed weight'].min():.4f} to {tilt['removed weight'].max():.4f}")
print(f"index turnover from the naive path's records: {row['mean benchmark turnover']:.8f}, recorded with the optimised paths: {float(tilt_rows['benchmark turnover'].iloc[0]):.8f}")
print(measures.T.to_string())

## 3. Tracking error: the naive tilt against each optimised path

The paired test compares the naive tilt's realised tracking error with each optimised path's on the same 566 months, with blocks of 12 months, 2,000 draws and the seed of `constants.py`, and Holm's correction across the seven comparisons. The interval is the test's 95 per cent interval on the difference.

In [ ]:
SETTINGS = dict(block=C.EVAL_BOOTSTRAP_BLOCK_MONTHS, draws=C.EVAL_BOOTSTRAP_DRAWS, level=C.EVAL_BOOTSTRAP_LEVEL, periods=C.PAIRED_PERIODS)
r1 = tilt["active_return"].to_numpy()
rows = []
for v in C.NAIVE_PATHS:
    s2 = active_paths.loc[months, f"{v} | C3 tilt only"].to_numpy(dtype=float)
    t = PT.test(r1, s2, seed=C.EVAL_BOOTSTRAP_SEED, **SETTINGS)
    rows.append({"path": f"{v} | C3 tilt only", "naive tilt (bp)": 1e4 * t["sd1"], "path (bp)": 1e4 * t["sd2"],
                 "path, committed table (bp)": 1e4 * float(tilt_rows.loc[v, "realised TE"]),
                 "difference (bp)": 1e4 * t["difference"], "standard error (bp)": 1e4 * t["se"], "critical value": t["critical"],
                 "lower (bp)": 1e4 * t["lower"], "upper (bp)": 1e4 * t["upper"],
                 "detectable difference (bp)": 1e4 * PT.detectable_difference(t["se"], t["critical"], C.PAIRED_DETECTABLE_POWER),
                 "correlation": t["correlation"], "p-value": t["p_value"]})
paired = pd.DataFrame(rows)
paired["Holm p-value"] = PT.holm(paired["p-value"].to_numpy())
paired.to_csv(out("naive_tilt_paired_tests"), index=False, float_format="%.8f")
print(paired[["path", "naive tilt (bp)", "path (bp)", "difference (bp)", "lower (bp)", "upper (bp)", "correlation", "Holm p-value"]].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"statistically significant at the 5 per cent level after Holm's correction: {int((paired['Holm p-value'] < C.NAIVE_SIGNIFICANCE).sum())} of {len(paired)}")

## 4. Turnover, costs and net active return

The naive tilt's active return, turnover and net active return at the two cost levels, beside the seven optimised paths' recorded figures, all in basis points a year except turnover, which is the mean one-way turnover a month.

In [ ]:
costs = pd.DataFrame({"realised TE (bp)": 1e4 * tilt_rows["realised TE"], "active return (gross) (bp)": 1e4 * tilt_rows["active return (gross)"],
                      "turnover (one-way)": tilt_rows["turnover (one-way)"],
                      "net active return at 50 bp (bp)": 1e4 * tilt_rows["net active return at 50 bp"],
                      "net active return at 100 bp (bp)": 1e4 * tilt_rows["net active return at 100 bp"]})
costs.index = [f"{v} | C3 tilt only" for v in costs.index]
nv = measures.loc["naive tilt"]
costs.loc["naive tilt"] = [1e4 * nv["realised TE"], 1e4 * nv["active return (gross)"], nv["turnover (one-way)"],
                           1e4 * nv["net active return at 50 bp"], 1e4 * nv["net active return at 100 bp"]]
costs.to_csv(out("naive_tilt_returns_and_costs"), float_format="%.6f")
print(costs.to_string(float_format=lambda x: f"{x:.4f}"))
print(f"months in which the naive tilt trades more than the cap: {int(nv['months above the turnover cap'])} of {len(months)}; "
      f"first month {nv['first month turnover']:.4f}, largest {nv['largest monthly turnover']:.4f} in {nv['month of the largest turnover']}")

## 5. Information ratios and realised betas

Each information ratio is given before costs and at the two cost levels, with its standard error from the same 2,000 block bootstrap resamples of the months; the cost is held at the full-sample figure in every resample. No test of a difference between two information ratios is made. The realised beta is the slope of each portfolio's monthly excess return on the benchmark's.

In [ ]:
def cost_of(table, label):
    return {bp: float(table.loc[label, f"cost at {bp} bp"]) for bp in COSTS}


ir_rows = [{"portfolio": "naive tilt", **N.information_ratios(r1, cost_of(measures, "naive tilt"), IDX)}]
for v in C.NAIVE_PATHS:
    ir_rows.append({"portfolio": f"{v} | C3 tilt only", **N.information_ratios(active_paths.loc[months, f"{v} | C3 tilt only"], cost_of(tilt_rows, v), IDX)})
information_ratios = pd.DataFrame(ir_rows).set_index("portfolio")
information_ratios.to_csv(out("naive_tilt_information_ratios"), float_format="%.8f")
beta_rows = [{"portfolio": "naive tilt", "realised beta": N.realised_beta(r1, bench_ex)}]
beta_rows += [{"portfolio": f"{v} | C3 tilt only", "realised beta": N.realised_beta(active_paths.loc[months, f"{v} | C3 tilt only"], bench_ex)} for v in C.NAIVE_PATHS]
realised_betas = pd.DataFrame(beta_rows)
realised_betas.to_csv(out("naive_tilt_realised_betas"), index=False, float_format="%.8f")
print(information_ratios.to_string(float_format=lambda x: f"{x:.4f}"))
print(realised_betas.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 6. Attribution to the six factors

The copied attribution splits each month's active return of the naive tilt into its six active exposures times the factors' returns, and a residual, with the betas of section 2. Each contribution is a figure per year with its standard error. The second table sets the naive tilt's contributions beside the seven optimised paths' recorded ones, in basis points a year.

In [ ]:
att = A.attribute(pd.DataFrame(W, index=months, columns=weights.columns), weights.loc[months], betas, P.factors_m, tilt["active_return"])
assert float(att["identity error"].abs().max()) <= C.ATTRIBUTION_IDENTITY_MAX_ABS_ERROR
att.to_csv(out("naive_tilt_attribution_monthly"), float_format="%.10f")
summary = A.summarise(att)
summary["portfolio"] = "naive tilt"
summary["targeted total (per year)"] = sum(summary[f"contribution {f} (per year)"] for f in TARGETED)
attribution = pd.DataFrame([summary]).set_index("portfolio")
attribution.to_csv(out("naive_tilt_attribution_summary"), float_format="%.10f")
print(attribution.T.to_string(float_format=lambda x: f"{x:.6f}"))
recorded, _ = source_table("attribution_summary")
recorded = recorded[recorded["set"] == "C3 tilt only"].set_index("variant").loc[list(C.NAIVE_PATHS)]
cols = [f"contribution {f} (per year)" for f in FACTORS] + ["factor total (per year)", "residual (per year)", "active return (per year)"]
beside = pd.concat([attribution[cols], recorded[cols].rename(index=lambda v: f"{v} | C3 tilt only")])
print((1e4 * beside).rename(columns=lambda c: c.replace(" (per year)", "")).to_string(float_format=lambda x: f"{x:.1f}"))

## 7. Two checks written after the results were known

Both checks were written on 27 November 2026, once the figures of sections 3 to 6 were known, to read the active return correctly. The first sets each difference between the naive tilt's information ratio and a path's beside the two ratios' standard errors, from the table of section 5; it is a comparison of sizes, and no test is made. The second rebuilds the naive tilt's active return from the spread: each month the active return equals the weight removed from Coal, Oil and Util times the spread, and the cell checks the identity and reports the spread's mean and standard deviation per year, its t-ratio with the months taken as independent, and its share of positive months.

In [ ]:
# Added after the run, 27 November 2026: the two checks of this section.
ir_table = pd.read_csv(out("naive_tilt_information_ratios"), index_col=0)
rows = []
for path in ir_table.index[1:]:
    for label, col, se in (("gross", "information ratio (gross)", "standard error (gross)"),
                           ("at 50 bp", "information ratio at 50 bp", "standard error at 50 bp"),
                           ("at 100 bp", "information ratio at 100 bp", "standard error at 100 bp")):
        d = ir_table.loc["naive tilt", col] - ir_table.loc[path, col]
        rows.append({"naive portfolio": "naive tilt", "path": path, "ratio": label, "naive": ir_table.loc["naive tilt", col],
                     "path value": ir_table.loc[path, col], "difference": d, "standard error, naive": ir_table.loc["naive tilt", se],
                     "standard error, path": ir_table.loc[path, se],
                     "difference above the larger standard error": bool(d > max(ir_table.loc["naive tilt", se], ir_table.loc[path, se]))})
ir_differences = pd.DataFrame(rows)
ir_differences.to_csv(out("naive_tilt_information_ratio_differences"), index=False, float_format="%.8f")
print(ir_differences.groupby("ratio", sort=False)["difference above the larger standard error"].sum().to_string())

Wm, Rm = weights.loc[months].copy(), returns_m.loc[months].copy()
Wm.columns = Rm.columns = industries
tilted = list(C.TILT_INDUSTRIES)
rest = [c for c in industries if c not in tilted]
spread = (Wm[rest] * Rm[rest]).sum(axis=1) / Wm[rest].sum(axis=1) - (Wm[tilted] * Rm[tilted]).sum(axis=1) / Wm[tilted].sum(axis=1)
removed = (1.0 - C.TILT_MAX_SHARE_OF_BENCHMARK) * Wm[tilted].sum(axis=1).to_numpy()
identity_error = float(np.abs(removed * spread.to_numpy() - r1).max())
assert identity_error < 1e-9
spread_table = pd.DataFrame([{"months": len(spread), "mean per year": 12 * spread.mean(), "standard deviation per year": spread.std(ddof=1) * np.sqrt(12),
                              "t-ratio": spread.mean() / (spread.std(ddof=1) / np.sqrt(len(spread))), "share of positive months": float((spread > 0).mean()),
                              "largest identity error": identity_error}])
spread_table.to_csv(out("naive_tilt_spread"), index=False, float_format="%.8f")
print(spread_table.to_string(index=False))

## What this notebook establishes, and what it does not settle

The naive tilt realises 105.6 basis points a year of tracking error from 1979-07 to 2026-08. The seven optimised paths realise 79.7 to 90.9, so the naive tilt realises 14.7 to 25.9 basis points a year more than each of them, each difference statistically significant at the 5 per cent level on the paired test after Holm's correction across the seven, with adjusted p-values of 0.0035 to 0.0115. The smallest difference is against the sample covariance on 120 months and the largest against RiskMetrics. The monthly active returns of the naive tilt and of each path are correlated at 0.76 to 0.86.

The naive tilt trades less. Its mean one-way turnover is 0.42% a month against 0.81% to 1.50% for the paths. It moves 11.7% of the portfolio in its first month, when it leaves the benchmark. It exceeds the turnover cap of 2% a month in 28 of the 566 months. The optimised paths exceed it only in the 2 or 3 months in which no portfolio could meet the cap and the tilt together. Its realised beta is 1.017, against 1.007 to 1.010 for the paths.

Over the sample the naive tilt earned an active return of +13.5 basis points a year before costs, against +1.8 to +6.9 for the paths, and +8.5 after a cost of 100 basis points per unit of turnover, against -14.0 to -2.8. No verdict was set on returns and no test of a difference in return was run, so these figures do not show that the naive tilt earns more than an optimised path, or that it earns less. Before costs each difference in information ratio, 0.05 to 0.11, is smaller than the standard error of either ratio, 0.12 to 0.14. The two checks of section 7, written once the results were known, read the gap in two parts:

- The naive tilt's active return is, month by month, the weight removed from Coal, Oil and Util times the spread. The spread averaged +1.22% a year over the 566 months, with a standard deviation of 15.28% a year and a t-ratio of 0.55. The optimised paths hedge part of that spread away, which is also why their tracking error is lower. How much the spread earned is a property of the period.
- After costs the gaps widen, because the naive tilt trades less than every path. That part is a property of the two methods: at 100 basis points per unit of turnover, the difference in information ratio exceeds the larger of the two standard errors against six of the seven paths.

The naive tilt's mean active exposures to value, profitability, investment and momentum are all below zero, -0.024, -0.013, -0.013 and -0.005: moving weight from Coal, Oil and Util to the other 46 industries lowers the portfolio's betas to all four.

**What this notebook does not settle.**

- The naive tilt is one construction of a tilt with no estimate of risk. Spreading the removed weight in another way, over a chosen set of industries close to the tilted ones for example, would realise a different tracking error, and none is run here.
- The figures are those of one sample and one vintage, 202608. A later vintage reproduces the comparisons, and not every digit, because French revises history.
- The optimised paths are read from risk-models-for-index-tracking at commit ddbb75c and one of them is rebuilt here. The others are taken as recorded, and their checks are in that repository.